# CV_02 — Transfert, robustesse et livraison

*Conception pédagogique : Mathieu Weill. Usage personnel d’apprentissage ; droits et crédits tiers conservés.*

**Compétition de travail : [catsdogs](https://www.kaggle.com/competitions/dogs-vs-cats-redux-kernels-edition)**

**Question : Faut-il adapter un gros modèle ou conserver une petite solution ?**

7 heures pédagogiques, hors pauses. Les exemples sont des jalons techniques courts ; l’essentiel du temps est consacré aux exercices, revues et expériences. **Version étudiant.**

**[Repère de compréhension : Évaluer la robustesse et livrer la vision](../../../course/chapitres/CV_02/README.md)**


## Préparation et limites

Lire README.md, compétitions et règles. Les données officielles sont téléchargées personnellement et placées dans data/competitions/<slug>/. Aucun remplacement silencieux par des données simulées. Poids préentraînés à préparer avant NLP_02/CV_01/CV_02. Les limites de calcul et d’échanges dépendent du concours et du compte.

## Notions du jour

La compétition de travail est Dogs vs. Cats Redux : vraies photos, classe positive chien, log-loss. Nous comparons un backbone gelé à un dernier bloc dégelé, sur le même split et le même budget. Contrôler BatchNorm, normalisation, augmentations et temps d’inférence. Le concours est historique et ne promet aucun prix actif.

Un run court ne désigne pas un vainqueur ; la solution finale demande des expériences plus longues justifiées, des ablations et une inférence reproductible. Aucun modèle ne doit être sélectionné sur les labels cachés du test.

In [ ]:
from pathlib import Path
import sys,os,json,time
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.competition import SPECS,read_competition,protocol,run_competition,binary_scores,get_folder
KEY='catsdogs';SPEC=SPECS[KEY]
QUICK=os.environ.get('MLLAB_QUICK','0')=='1'
LIMIT=int(os.environ.get('MLLAB_LIMIT','1000' if QUICK else '15000'))
IMAGE_LIMIT=int(os.environ.get('MLLAB_IMAGE_LIMIT','24' if QUICK else '240'))
plt.rcParams.update({'figure.figsize':(10,4),'axes.spines.top':False,'axes.spines.right':False})
print('Compétition :',SPEC['slug'],'métrique :',SPEC['metric'],'sens :',SPEC['direction'])
if KEY!='catsdogs':a,test,sample,protocol_record=protocol(KEY)
else:
 from mllab.vision_competition import image_protocol
 image_protocol(limit=IMAGE_LIMIT)
print('Données officielles trouvées. Profil rapide :',QUICK,'; aucun score Kaggle inventé.')

## Comparer tête seule et adaptation partielle

In [ ]:
from mllab.vision_competition import run_images
rows=[];last_run=None
for mode in ['head','partial']:
 out,scores,hist=run_images(limit=IMAGE_LIMIT,epochs=1 if QUICK else 3,mode=mode,augment=True,allow_download=False)
 manifest=json.loads((out/'manifest.json').read_text());metrics=json.loads((out/'metrics.json').read_text())
 rows.append({'mode':mode,'n_train':manifest['n_train'],'n_valid':manifest['n_valid'],'trainable':manifest['trainable'],'seconds':metrics['seconds'],**scores});last_run=out
display(pd.DataFrame(rows))
print('Augmentation train uniquement. Une seule epoch et peu d’images = vérification technique, pas solution optimisée.')

## Contrôler les probabilités et préparer l’inférence officielle

In [ ]:
pred=pd.read_csv(last_run/'predictions.csv')
assert pred.probability_dog.between(0,1).all()
print('Après choix du checkpoint, pour générer les probabilités du vrai test :')
print('from mllab.vision_competition import submit_images')
print('submit_images(last_run)')
print('L’inférence de tout le test n’est pas déclenchée automatiquement pendant ce TP court.')
print('Concours historique clos : vérifier la disponibilité des soumissions tardives ; aucun prix actif promis.')

## Atelier autonome : produire des preuves

1. Mesurer une inférence sur un batch, en précisant machine, batch, prétraitement et warm-up ; ne pas comparer des mesures de matériels différents sans le dire.
2. Rédiger une fiche modèle et une démonstration qui refuse une entrée invalide.
3. Restituer le projet : une amélioration conservée, une abandonnée, un risque non résolu et une contribution individuelle.

Pour chaque essai : hypothèse avant calcul, code/configuration, empreinte du split, métrique officielle, durée, budget, conclusion, prochaine expérience. Chaque membre doit pouvoir expliquer son apport.

In [ ]:
HYPOTHESE = 'À écrire avant le prochain calcul'
CONCLUSION = 'À compléter avec une preuve et une limite'
# Ajouter ici une expérience limitée, puis expliquer la décision de la conserver ou non.

## Questions de sortie

- Pourquoi un préentraînement ImageNet ne garantit-il pas de bons résultats sur toutes les photos de chiens et de chats ?
- À quel moment le test final cesse-t-il d’être final ?

**Remise :** un run traçable, une conclusion, un artefact ou une soumission valide et une prochaine hypothèse. Aucun score public ni prix garanti.